# AI‑Augmented Credit Risk Model

In [1]:
import pandas as pd

df = pd.read_csv("credit_portfolio_clean.csv")
df.head()

,customer_id,age,annual_income_gbp,employment_status,employment_length_years,loan_type,loan_amount_gbp,loan_term_months,interest_rate_pct,credit_score,outstanding_balance_gbp,days_past_due,previous_defaults,default_flag,loan_to_income_ratio,dpd_bucket,risk_bucket
0,100000,43,73100.0,Employed,1.8,Unsecured Personal Loan,25400,36,5.74,845.0,19000,90,0,0,0.347469,90+ DPD,High Risk
1,100001,37,57700.0,Employed,4.1,Unsecured Personal Loan,19200,12,5.64,565.0,13100,10,0,0,0.332756,Current,Medium Risk
2,100002,44,37400.0,Employed,8.1,Unsecured Personal Loan,53200,24,4.77,639.0,8000,30,1,0,1.422460,30-59 DPD,High Risk
3,100003,53,NaN,Employed,0.7,Mortgage,5500,36,12.55,670.0,5700,0,0,0,NaN,Current,Low Risk
4,100004,36,51500.0,Contract,4.3,Auto Loan,28400,60,10.03,715.0,8300,0,0,0,0.551456,Current,Low Risk


In [2]:
# Removed ID column
df_model = df.drop(columns=["customer_id"])

# Target
y = df_model["default_flag"]

# Features
X = df_model.drop("default_flag", axis=1)

In [3]:
# Converting text → numbers
X = pd.get_dummies(X, drop_first=True)

In [4]:
from sklearn.impute import SimpleImputer

# Median imputation
imputer = SimpleImputer(strategy="median")

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

In [5]:
X.isna().sum().sum()

np.int64(0)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

C:\Users\karun\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [8]:
from sklearn.metrics import roc_auc_score

y_pred = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.731608025588834


In [9]:
# Coefficients
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.coef_[0]
})

# Sorting by absolute importance
coefficients["Abs_Importance"] = coefficients["Importance"].abs()
coefficients = coefficients.sort_values(by="Abs_Importance", ascending=False)

coefficients.head(10)

,Feature,Importance,Abs_Importance
9,previous_defaults,0.429720,0.429720
21,risk_bucket_Medium Risk,-0.154195,0.154195
11,employment_status_Employed,0.115638,0.115638
20,risk_bucket_Low Risk,-0.085490,0.085490
17,dpd_bucket_60-89 DPD,0.076309,0.076309
5,interest_rate_pct,0.071318,0.071318
16,loan_type_Unsecured Personal Loan,0.045384,0.045384
14,employment_status_Unemployed,-0.038946,0.038946
10,loan_to_income_ratio,-0.037636,0.037636
18,dpd_bucket_90+ DPD,-0.022883,0.022883


In [10]:
# probability of default for all data
df["pd_score"] = model.predict_proba(X)[:, 1]

df.head()

,customer_id,age,annual_income_gbp,employment_status,employment_length_years,loan_type,loan_amount_gbp,loan_term_months,interest_rate_pct,credit_score,outstanding_balance_gbp,days_past_due,previous_defaults,default_flag,loan_to_income_ratio,dpd_bucket,risk_bucket,pd_score
0,100000,43,73100.0,Employed,1.8,Unsecured Personal Loan,25400,36,5.74,845.0,19000,90,0,0,0.347469,90+ DPD,High Risk,0.074947
1,100001,37,57700.0,Employed,4.1,Unsecured Personal Loan,19200,12,5.64,565.0,13100,10,0,0,0.332756,Current,Medium Risk,0.059043
2,100002,44,37400.0,Employed,8.1,Unsecured Personal Loan,53200,24,4.77,639.0,8000,30,1,0,1.422460,30-59 DPD,High Risk,0.241471
3,100003,53,NaN,Employed,0.7,Mortgage,5500,36,12.55,670.0,5700,0,0,0,NaN,Current,Low Risk,0.050416
4,100004,36,51500.0,Contract,4.3,Auto Loan,28400,60,10.03,715.0,8300,0,0,0,0.551456,Current,Low Risk,0.029376


In [11]:
# Model-based risk categories
def assign_risk(pd):
    if pd > 0.4:
        return "High Risk"
    elif pd > 0.2:
        return "Medium Risk"
    else:
        return "Low Risk"

df["ai_risk_bucket"] = df["pd_score"].apply(assign_risk)

In [12]:
pd.crosstab(df["risk_bucket"], df["ai_risk_bucket"])

ai_risk_bucket,High Risk,Low Risk,Medium Risk
risk_bucket,,,
High Risk,14,237,45
Low Risk,0,376,0
Medium Risk,0,322,6


In [13]:
df.groupby("ai_risk_bucket")["default_flag"].mean()

ai_risk_bucket
High Risk      0.428571
Low Risk       0.062032
Medium Risk    0.274510
Name: default_flag, dtype: float64

In [14]:
df.to_csv("final_credit_risk_output.csv", index=False)